# Subgraphs

## Review

So far, we've built graphs where every node lived in one top-level workflow.

## Goals

**Subgraphs** let us package a smaller graph and reuse it inside a larger graph. This matters because it enables:

1. **Modular development** — teams can build one part of a workflow independently
2. **Reusable components** — the same graph can be plugged into multiple parents
3. **Multi-agent systems** — each specialist can be its own graph with its own tools and logic

Think of a subgraph as a function made out of graph nodes:

```
Parent Graph
    ├── Node A
    ├── Subgraph B
    │      ├── Node B1
    │      └── Node B2
    └── Node C
```

This notebook covers the two main integration patterns:
- **Add a compiled subgraph directly as a node** when the parent and child share state keys
- **Call a subgraph inside a wrapper node** when the parent and child use different state schemas

In [ ]:
%run ../../langchain_common.py

In [ ]:
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from typing import Literal
from typing_extensions import TypedDict
from IPython.display import Image, display

## Pattern 1: Add Subgraph as a Node (Shared State)

If the parent graph and child graph share the same state keys, the simplest pattern is:

1. Build the child graph with `StateGraph`
2. Compile it
3. Pass the compiled graph directly to `add_node(...)` in the parent

Here, both subgraphs use `MessagesState`, so they share the `messages` key with the parent graph.

In [ ]:
research_prompt = SystemMessage(
    content="You are a research assistant. Answer the user's question with three concrete points. Be concise."
)
summary_prompt = SystemMessage(
    content="You are a summarizer. Read the conversation so far and turn the latest answer into exactly two bullet points."
)


def research_step(state: MessagesState):
    response = llm_noreason.invoke([research_prompt] + state["messages"])
    return {"messages": [response]}


def summarize_step(state: MessagesState):
    response = llm_noreason.invoke([summary_prompt] + state["messages"])
    return {"messages": [response]}


research_builder = StateGraph(MessagesState)
research_builder.add_node("researcher", research_step)
research_builder.add_edge(START, "researcher")
research_builder.add_edge("researcher", END)
research_subgraph = research_builder.compile(name="research_subgraph")

summarizer_builder = StateGraph(MessagesState)
summarizer_builder.add_node("summarizer", summarize_step)
summarizer_builder.add_edge(START, "summarizer")
summarizer_builder.add_edge("summarizer", END)
summarizer_subgraph = summarizer_builder.compile(name="summarizer_subgraph")

In [ ]:
shared_state_builder = StateGraph(MessagesState)
shared_state_builder.add_node("research_team", research_subgraph)
shared_state_builder.add_node("summarizer_team", summarizer_subgraph)
shared_state_builder.add_edge(START, "research_team")
shared_state_builder.add_edge("research_team", "summarizer_team")
shared_state_builder.add_edge("summarizer_team", END)

shared_state_graph = shared_state_builder.compile(name="shared_state_graph")

display(Image(shared_state_graph.get_graph(xray=1).draw_mermaid_png()))

In [ ]:
shared_result = shared_state_graph.invoke(
    {
        "messages": [
            HumanMessage(content="What are the main benefits of LangGraph subgraphs?")
        ]
    }
)

for message in shared_result["messages"]:
    print(f"{message.type.upper()}: {message.content}\n")

## Pattern 2: Call Subgraph Inside a Node (Different State)

Sometimes the parent graph and child graph do **not** share the same schema.

- Parent state uses `foo`
- Child state uses `bar`

In that case, the parent cannot plug the child in directly. Instead, a wrapper node:

1. Transforms parent state into child state
2. Invokes the subgraph
3. Maps the child result back into the parent state

In [ ]:
class ParentState(TypedDict):
    foo: str
    child_input: str
    child_output: str


class ChildState(TypedDict):
    bar: str


def child_transform(state: ChildState):
    return {"bar": f"Child graph processed: {state['bar'].upper()}"}


child_builder = StateGraph(ChildState)
child_builder.add_node("child_transform", child_transform)
child_builder.add_edge(START, "child_transform")
child_builder.add_edge("child_transform", END)
child_graph = child_builder.compile(name="child_graph")


def call_child_graph(state: ParentState):
    child_state = {"bar": f"from foo -> {state['foo']}"}
    child_result = child_graph.invoke(child_state)
    return {
        "child_input": child_state["bar"],
        "child_output": child_result["bar"],
    }

In [ ]:
wrapper_builder = StateGraph(ParentState)
wrapper_builder.add_node("call_child_graph", call_child_graph)
wrapper_builder.add_edge(START, "call_child_graph")
wrapper_builder.add_edge("call_child_graph", END)

wrapper_graph = wrapper_builder.compile(name="wrapper_graph")

wrapper_result = wrapper_graph.invoke(
    {
        "foo": "subgraphs make large systems easier to reason about",
        "child_input": "",
        "child_output": "",
    }
)

print("Parent foo:", wrapper_result["foo"])
print("Sent to child as bar:", wrapper_result["child_input"])
print("Returned from child as bar:", wrapper_result["child_output"])

## Practical Example: Multi-Agent with Subgraphs

A common real-world use case is a **supervisor graph** coordinating specialist agents.

We'll build:

- a **research agent** subgraph with a fact lookup tool
- a **math agent** subgraph with a calculator tool
- a **supervisor** node that routes the user request to the right subgraph

Each specialist stays modular, which makes the system easier to test and extend.

In [ ]:
@tool
def lookup_topic(topic: str) -> str:
    """Look up a short fact about a topic."""
    facts = {
        "langgraph": "LangGraph is a framework for building stateful, multi-step applications with graphs.",
        "subgraph": "A subgraph is a compiled graph reused inside a larger parent graph.",
        "databricks": "Databricks provides model-serving endpoints that work with OpenAI-compatible clients.",
    }
    for key, value in facts.items():
        if key in topic.lower():
            return value
    return f"No stored fact for '{topic}'. Try langgraph, subgraph, or databricks."


@tool
def calculate(expression: str) -> str:
    """Evaluate a mathematical expression."""
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"


def run_specialist(state: MessagesState, system_prompt: str, tools: list):
    llm_with_tools = llm_noreason.bind_tools(tools)
    response = llm_with_tools.invoke([SystemMessage(content=system_prompt)] + state["messages"])

    if not response.tool_calls:
        return {"messages": [response]}

    tool_map = {tool_.name: tool_ for tool_ in tools}
    tool_messages = []

    for tool_call in response.tool_calls:
        result = tool_map[tool_call["name"]].invoke(tool_call["args"])
        tool_messages.append(ToolMessage(content=str(result), tool_call_id=tool_call["id"]))

    final = llm_noreason.invoke(
        [SystemMessage(content=system_prompt)] + state["messages"] + [response] + tool_messages
    )
    return {"messages": [final]}


def research_agent_node(state: MessagesState):
    return run_specialist(
        state,
        "You are a research specialist. Use lookup_topic when the user needs facts or explanations. Be concise.",
        [lookup_topic],
    )


def math_agent_node(state: MessagesState):
    return run_specialist(
        state,
        "You are a math specialist. Use calculate for arithmetic or expressions. Explain the result briefly.",
        [calculate],
    )

In [ ]:
research_agent_builder = StateGraph(MessagesState)
research_agent_builder.add_node("research_agent_node", research_agent_node)
research_agent_builder.add_edge(START, "research_agent_node")
research_agent_builder.add_edge("research_agent_node", END)
research_agent_graph = research_agent_builder.compile(name="research_agent_graph")

math_agent_builder = StateGraph(MessagesState)
math_agent_builder.add_node("math_agent_node", math_agent_node)
math_agent_builder.add_edge(START, "math_agent_node")
math_agent_builder.add_edge("math_agent_node", END)
math_agent_graph = math_agent_builder.compile(name="math_agent_graph")


class SupervisorState(MessagesState):
    next_agent: Literal["research_agent", "math_agent"]


router_prompt = SystemMessage(
    content="""You are a supervisor.
Choose the best specialist for the user's latest request.
Respond with ONLY one of these labels:
- research_agent
- math_agent"""
)


def supervisor_router(state: SupervisorState):
    response = llm_noreason.invoke([router_prompt] + state["messages"])
    route = response.content.strip().lower()
    next_agent = "math_agent" if "math" in route else "research_agent"
    return {"next_agent": next_agent}


def choose_specialist(state: SupervisorState) -> Literal["research_agent", "math_agent"]:
    return state["next_agent"]


supervisor_builder = StateGraph(SupervisorState)
supervisor_builder.add_node("supervisor", supervisor_router)
supervisor_builder.add_node("research_agent", research_agent_graph)
supervisor_builder.add_node("math_agent", math_agent_graph)
supervisor_builder.add_edge(START, "supervisor")
supervisor_builder.add_conditional_edges(
    "supervisor",
    choose_specialist,
    {
        "research_agent": "research_agent",
        "math_agent": "math_agent",
    },
)
supervisor_builder.add_edge("research_agent", END)
supervisor_builder.add_edge("math_agent", END)

supervisor_graph = supervisor_builder.compile(name="supervisor_graph")

display(Image(supervisor_graph.get_graph(xray=1).draw_mermaid_png()))

In [ ]:
research_demo = supervisor_graph.invoke(
    {
        "messages": [
            HumanMessage(content="What is LangGraph and why are subgraphs useful?")
        ]
    }
)

print("Research route:", research_demo["next_agent"])
print(research_demo["messages"][-1].content)
print()

math_demo = supervisor_graph.invoke(
    {
        "messages": [
            HumanMessage(content="What is (18 + 6) / 4?")
        ]
    }
)

print("Math route:", math_demo["next_agent"])
print(math_demo["messages"][-1].content)

## Subgraph Persistence

Subgraphs also interact with **checkpointing**:

- By default, a subgraph **inherits the parent's checkpointer for that invocation**
- If you compile a subgraph with `checkpointer=True`, it gets its **own checkpoint namespace for the thread**

This matters when you want to pause execution and inspect nested state.

We'll use `interrupt_before=["child_graph"]` so the parent pauses right before entering the child subgraph. Then we can call:

```python
graph.get_state(config, subgraphs=True)
```

to inspect the pending subgraph state.

In [ ]:
class CounterState(TypedDict):
    count: int


def double_count(state: CounterState):
    return {"count": state["count"] * 2}


def increment_count(state: CounterState):
    return {"count": state["count"] + 1}


def build_child_graph(checkpointer=None):
    child_builder = StateGraph(CounterState)
    child_builder.add_node("increment_count", increment_count)
    child_builder.add_edge(START, "increment_count")
    child_builder.add_edge("increment_count", END)
    return child_builder.compile(checkpointer=checkpointer, name="child_graph")


def build_parent_graph(child_graph):
    parent_builder = StateGraph(CounterState)
    parent_builder.add_node("double_count", double_count)
    parent_builder.add_node("child_graph", child_graph)
    parent_builder.add_edge(START, "double_count")
    parent_builder.add_edge("double_count", "child_graph")
    parent_builder.add_edge("child_graph", END)
    return parent_builder.compile(
        checkpointer=MemorySaver(),
        interrupt_before=["child_graph"],
        name="parent_graph",
    )

In [ ]:
per_invocation_graph = build_parent_graph(build_child_graph())
per_thread_graph = build_parent_graph(build_child_graph(checkpointer=True))

config_once = {"configurable": {"thread_id": "subgraph-once"}}
config_thread = {"configurable": {"thread_id": "subgraph-thread"}}

per_invocation_graph.invoke({"count": 3}, config=config_once)
per_thread_graph.invoke({"count": 3}, config=config_thread)

snapshot_once = per_invocation_graph.get_state(config_once, subgraphs=True)
snapshot_thread = per_thread_graph.get_state(config_thread, subgraphs=True)

once_subgraph_state = snapshot_once.tasks[0].state
thread_subgraph_state = snapshot_thread.tasks[0].state

print("Per-invocation child namespace:")
print(once_subgraph_state.config["configurable"]["checkpoint_ns"])
print()

print("Per-thread child namespace:")
print(thread_subgraph_state.config["configurable"]["checkpoint_ns"])
print()

print("Nested state returned by get_state(..., subgraphs=True):")
print(thread_subgraph_state)

final_state = per_thread_graph.invoke(None, config=config_thread)
print()
print("After resuming the paused parent graph:", final_state)

## Key Takeaways

| Idea | When to use it | What to remember |
| --- | --- | --- |
| Add subgraph as node | Parent and child share keys like `messages` | Compile the child graph, then pass it directly to `add_node(...)` |
| Wrapper node pattern | Parent and child use different schemas | Transform parent state → child state → parent state |
| Multi-agent subgraphs | Specialists need isolated tools and logic | Each agent can be its own reusable graph |
| Checkpointer inheritance | You need persistence or inspection | Default is per-invocation inheritance; `checkpointer=True` gives the child its own thread-level namespace |
| `get_state(..., subgraphs=True)` | You want to inspect nested execution state | Useful when a parent graph is paused around a subgraph boundary |